In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.interpolate import griddata
import os

In [ ]:
mesh_size = 2.0
normal_stress = 10.0
Spring_stiffness = 3000

DATA_FOLDER = f'./Stiffness/Y{Spring_stiffness}/ShearFace'
RUPTURE_POSITIONS = [5, 15, 25, 35, 45, 55, 65, 75, 80, 85, 90, 95, 100, 105, 110, 115]

In [ ]:
GIF_FOLDER = os.path.join(DATA_FOLDER, 'GIFS')
os.makedirs(GIF_FOLDER, exist_ok=True)


all_s12_data = []
all_mu_midline_data = []
all_positions = []

# 第一輪：收集所有數據
print("Collecting data for GIF generation...")

for RUPTURE_POSITION in RUPTURE_POSITIONS:
    npz_file = os.path.join(DATA_FOLDER, f'ShearFace-{RUPTURE_POSITION}.npz')
    
    if not os.path.exists(npz_file):
        print(f"File {npz_file} not found, skipping...")
        continue
    
    print(f"Loading {npz_file}...")
    
    # 載入數據
    data = np.load(npz_file)
    x = data['x']
    y = data['y']
    z = data['z']
    s11 = data['s11']
    s12 = data['s12']
    mu = data['mu']
    
    # === 方格網插值設定 ===
    # 設定網格解析度（可調整）
    grid_resolution_y = 400  # Y方向網格點數
    grid_resolution_z = 200  # Z方向網格點數
    
    # 創建規則網格
    y_min, y_max = y.min(), y.max()
    z_min, z_max = z.min(), z.max()
    
    yi = np.linspace(y_min, y_max, grid_resolution_y)
    zi = np.linspace(z_min, z_max, grid_resolution_z)
    Yi, Zi = np.meshgrid(yi, zi)
    
    # 插值 S12 到網格
    points = np.column_stack((y, z))
    s12_grid = griddata(points, s12, (Yi, Zi), method='linear')
    
    # 存儲 S12 heatmap 數據
    all_s12_data.append({
        'Yi': Yi,
        'Zi': Zi,
        's12_grid': s12_grid,
        'position': RUPTURE_POSITION,
        'y_range': (y_min, y_max),
        'z_range': (z_min, z_max)
    })
    
    # === Z 中線分析（與原代碼相同邏輯）===
    NUM_Y_BINS = 100
    INITIAL_TOL_RATIO_Z = 0.01
    MAX_TOL_RATIO_Z = 0.05
    MIN_REQUIRED_POINTS_Z = 50
    
    z_mid = 0.5 * (z_min + z_max)
    z_span = z_max - z_min
    tol_ratio_z = INITIAL_TOL_RATIO_Z
    
    # 尋找 Z 中線附近的點
    for _ in range(5):
        tol_z = tol_ratio_z * z_span
        mid_mask_z = np.abs(z - z_mid) <= tol_z
        if np.count_nonzero(mid_mask_z) >= MIN_REQUIRED_POINTS_Z or tol_ratio_z >= MAX_TOL_RATIO_Z:
            break
        tol_ratio_z = min(tol_ratio_z * 2, MAX_TOL_RATIO_Z)
    
    # Y 方向分箱計算 midline mu
    y_edges_mid = np.linspace(y_min, y_max, NUM_Y_BINS + 1)
    y_centers_mid = 0.5 * (y_edges_mid[:-1] + y_edges_mid[1:])
    mu_midline_y = np.full(NUM_Y_BINS, np.nan)
    
    if np.count_nonzero(mid_mask_z) > 0:
        counts_y = np.zeros(NUM_Y_BINS, dtype=int)
        for yy, mm in zip(y[mid_mask_z], mu[mid_mask_z]):
            if not np.isnan(mm):  # 只處理有效的 mu 值
                bi = np.searchsorted(y_edges_mid, yy, side='right') - 1
                if 0 <= bi < NUM_Y_BINS:
                    if np.isnan(mu_midline_y[bi]):
                        mu_midline_y[bi] = 0.0
                    mu_midline_y[bi] += mm
                    counts_y[bi] += 1
        good_y = counts_y > 0
        mu_midline_y[good_y] /= counts_y[good_y]
    else:
        # Fallback：每個 Y-bin 取最接近 Z-mid 的點
        for bi in range(NUM_Y_BINS):
            in_bin = (y >= y_edges_mid[bi]) & (y < y_edges_mid[bi+1])
            if not np.any(in_bin):
                continue
            valid_mu = mu[in_bin]
            valid_z = z[in_bin]
            valid_mask = ~np.isnan(valid_mu)
            if np.any(valid_mask):
                idx_local = np.argmin(np.abs(valid_z[valid_mask] - z_mid))
                mu_midline_y[bi] = valid_mu[valid_mask][idx_local]
    
    # 存儲 midline mu 數據
    all_mu_midline_data.append({
        'y_centers': y_centers_mid,
        'mu_midline': mu_midline_y,
        'position': RUPTURE_POSITION,
        'z_mid': z_mid,
        'tol_ratio': tol_ratio_z
    })
    
    all_positions.append(RUPTURE_POSITION)

# === 創建 S12 Heatmap GIF ===
print("\nCreating S12 Heatmap GIF...")

if all_s12_data:
    # 找到所有數據的色階範圍
    all_s12_values = np.concatenate([d['s12_grid'][~np.isnan(d['s12_grid'])] for d in all_s12_data])
    vmin, vmax = np.percentile(all_s12_values, [1, 99])  # 用1%和99%來避免極值影響
    
    # 找到統一的座標範圍
    y_global_min = min(d['y_range'][0] for d in all_s12_data)
    y_global_max = max(d['y_range'][1] for d in all_s12_data)
    z_global_min = min(d['z_range'][0] for d in all_s12_data)
    z_global_max = max(d['z_range'][1] for d in all_s12_data)
    
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    
    def animate_s12(frame):
        ax.clear()
        data = all_s12_data[frame]
        
        im = ax.contourf(data['Yi'], data['Zi'], data['s12_grid'], 
                        levels=50, cmap='RdBu_r', vmin=vmin, vmax=vmax)
        
        ax.set_xlabel('Y Position')
        ax.set_ylabel('Z Position')
        ax.set_title(f'S12 - Position {data["position"]}, Displacement Control, {normal_stress:.1f}MPa, E = {Spring_stiffness}MPa, {mesh_size}mm')
        ax.set_xlim(y_global_min, y_global_max)
        ax.set_ylim(z_global_min, z_global_max)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        
        return [im]
    
    # 創建動畫
    anim_s12 = FuncAnimation(fig, animate_s12, frames=len(all_s12_data), interval=500, blit=False, repeat=True)
    
    # 保存 GIF
    gif_path_s12 = os.path.join(GIF_FOLDER, 'S12_Heatmap_Animation.gif')
    anim_s12.save(gif_path_s12, writer=PillowWriter(fps=2), dpi = 300)
    plt.close(fig)
    print(f"S12 Heatmap GIF saved: {gif_path_s12}")

# === 創建 Midline Mu GIF ===
print("Creating Midline Mu GIF...")

if all_mu_midline_data:
    # 找到 mu 的色階範圍
    all_mu_values = []
    for d in all_mu_midline_data:
        valid_mu = d['mu_midline'][~np.isnan(d['mu_midline'])]
        if len(valid_mu) > 0:
            all_mu_values.extend(valid_mu)
    
    if all_mu_values:
        mu_min, mu_max = np.percentile(all_mu_values, [1, 99])
        
        # 找到統一的 Y 範圍
        y_global_min = min(d['y_centers'].min() for d in all_mu_midline_data)
        y_global_max = max(d['y_centers'].max() for d in all_mu_midline_data)
        
        fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
        
        def animate_mu(frame):
            ax.clear()
            data = all_mu_midline_data[frame]
            
            # 過濾有效數據
            valid_mask = ~np.isnan(data['mu_midline'])
            y_valid = data['y_centers'][valid_mask]
            mu_valid = data['mu_midline'][valid_mask]
            
            if len(mu_valid) > 0:
                ax.plot(y_valid, mu_valid, 'o-', linewidth=2, markersize=4, color = "blue")
                ax.axhline(y = 0.7, color = "red", linestyle = "--")
            
            ax.set_xlabel('Y Position')
            ax.set_ylabel('Friction Coefficient μ')
            ax.set_title(f'Midline μ (Z={data["z_mid"]:.1f}) - Position {data["position"]}, Displacement Control, {normal_stress:.1f}MPa, E = {Spring_stiffness}MPa, {mesh_size}mm')
            ax.set_xlim(y_global_min, y_global_max)
            ax.set_ylim(mu_min, mu_max)
            ax.grid(True, alpha=0.3)
            
            return ax.lines
        
        # 創建動畫
        anim_mu = FuncAnimation(fig, animate_mu, frames=len(all_mu_midline_data), interval=500, blit=False, repeat=True)
        
        # 保存 GIF
        gif_path_mu = os.path.join(GIF_FOLDER, 'Midline_Mu_Animation.gif')
        anim_mu.save(gif_path_mu, writer=PillowWriter(fps=2), dpi = 300)
        plt.close(fig)
        print(f"Midline Mu GIF saved: {gif_path_mu}")

print(f"\nAll GIF files saved in: {GIF_FOLDER}/")
print(f"Processed {len(all_positions)} positions: {all_positions}")

./Stiffness/Y3000/ShearFace/GIFS
Loading ./Stiffness/Y3000/ShearFace/ShearFace-5.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-15.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-25.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-35.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-45.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-55.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-65.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-75.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-80.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-85.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-90.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-95.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-100.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-105.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-110.npz...
Loading ./Stiffness/Y3000/ShearFace/ShearFace-115.npz...

Creating S12 Heatmap GIF...
S12 Heatmap GIF saved: ./Stiffness/Y300